# 🔬 Laboratorio del Modelo — Escasez y Resiliencia
## `V(t) = Capital × (1+r)^t × Φ(t)`

**Autor:** Jose Antonio Vilar · **Versión:** 1.0 · **Fecha:** Marzo 2026

---

Este notebook es el laboratorio matemático del paper *"Escasez y Resiliencia: Un Marco Matemático para Invertir en la Era de la Autorreplicación Artificial"*.

### Estructura
| # | Sección | Objetivo |
|---|---------|----------|
| 1 | Setup y datos | Cargar parámetros del portfolio real |
| 2 | Fundamentos del modelo | Propiedades del modelo logístico |
| 3 | Modelos alternativos | Logística, Gompertz, Régimen Dual |
| 4 | Calibración de parámetros | r desde SOX, λ desde cadencia LLM |
| 5 | Comparativa de curvas | Visualización y tabla de propiedades |
| 6 | Backtesting 2018–2025 | Validación empírica con datos reales |
| 7 | Análisis de sensibilidad | ¿Cuánto importa cada parámetro? |
| 8 | Detector SOX/Cobre | Simulación del cambio de paradigma |
| 9 | Proyecciones finales | Escenarios Base / Acelerado / Óptimo |


## 1. Setup y parámetros

| Celda | Sección | Contenido |
|-------|---------|----------|
| 2–3   | Setup y datos | Cargar parámetros del portfolio real |
| 4–5   | §2 Fundamentos | Modelo logístico y sus propiedades |
| 6–8   | §3 Factor Φ_L | Definición, parámetros, visualización |
| 9–11  | §4 Calibración | r desde SOX, γ desde cadencia LLM |
| 12–13 | §5 Propiedades | Tabla de propiedades del modelo |
| 14–17 | §6 Backtesting | Validación con datos históricos reales |
| 18–19 | §7 Sensibilidad | Impacto de cada parámetro |
| 20–21 | §8 SOX/Cobre | Simulación del detector de régimen |
| 22–23 | §9 Proyecciones | 3 escenarios Base/Acelerado/Óptimo |
| 24   | §10 Conclusiones | Resumen del modelo y referencias |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

# ── Estilo oscuro del portfolio ──────────────────────────────────
BG    = '#0d1117'; PANEL = '#161b22'; GRID = '#21262d'
WHITE = '#e6edf3'; MUTED = '#8b949e'
C_EXP  = '#ff7b72'; C_LOG = '#79c0ff'; C_GOM = '#d2a8ff'
C_DUAL = '#56d364'; C_BASE = '#e3b341'; C_OPT = '#ffa657'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': PANEL,
    'axes.edgecolor': GRID, 'axes.labelcolor': WHITE,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'text.color': WHITE, 'grid.color': GRID,
    'grid.linewidth': 0.6, 'legend.facecolor': '#1c2128',
    'legend.edgecolor': GRID, 'font.family': 'DejaVu Sans',
})

print("✅ Entorno configurado")


In [ ]:
# ── Parámetros del portfolio (portfolio.yaml) ────────────────────
import yaml, os

YAML_PATH = os.path.join(os.path.dirname(os.getcwd()),
            'mnt', 'Cartera-Inversion', 'portfolio.yaml') \
            if os.path.exists('/sessions') else 'portfolio.yaml'

try:
    with open(YAML_PATH) as f:
        portfolio = yaml.safe_load(f)
    tesis = portfolio.get('tesis', {})
    esc   = tesis.get('escenarios', {})
    B_r   = float(esc.get('base',      {}).get('r',   0.20))
    B_lam = float(esc.get('base',      {}).get('lam', 0.05))
    A_r   = float(esc.get('acelerado', {}).get('r',   0.25))
    A_lam = float(esc.get('acelerado', {}).get('lam', 0.15))
    O_r   = float(esc.get('optimo',    {}).get('r',   0.30))
    O_lam = float(esc.get('optimo',    {}).get('lam', 0.30))
    ultima_rev = tesis.get('ultima_revision', 'N/A')
    print(f"✅ portfolio.yaml cargado | Última revisión: {ultima_rev}")
except Exception as e:
    print(f"⚠️  No se pudo cargar portfolio.yaml ({e}) — usando valores por defecto")
    B_r, B_lam = 0.20, 0.05
    A_r, A_lam = 0.25, 0.15
    O_r, O_lam = 0.30, 0.30

print(f"   Base:      r={B_r:.2f}  λ={B_lam:.2f}")
print(f"   Acelerado: r={A_r:.2f}  λ={A_lam:.2f}")
print(f"   Óptimo:    r={O_r:.2f}  λ={O_lam:.2f}")


## 2. Fundamentos matemáticos del modelo

### 2.1 El modelo Escasez y Resiliencia

El modelo es:
$$V(t) = Capital \times (1+r)^t \times \frac{\Phi_L(t)}{\Phi_L(0)}$$

donde $\Phi_L(t) = 1 + \frac{K}{1 + e^{-\gamma(t - t_0)}}$ es la función logística de autorreplicación (Verhulst, 1838).

La normalización por $\Phi_L(0)$ garantiza que $V(0) = Capital$ para cualquier combinación de parámetros.

### 2.2 Propiedades fundamentales

1. **Saturación natural**: $\Phi_L(t) \to 1 + K$ cuando $t \to \infty$ — el crecimiento tiene techo $K$
2. **Falsificabilidad**: si la adopción de IA se detiene, $\Phi_L(t) \to 1$ y el modelo se reduce a crecimiento compuesto estándar $(1+r)^t$
3. **Punto de inflexión**: la máxima aceleración ocurre en $t = t_0$ — calibrable empíricamente desde la correlación SOX/Cobre


In [ ]:
# ── Visualización de las propiedades del modelo logístico ───────────────
t = np.linspace(0, 10, 300)

# Escenarios del paper
escenarios = [
    ('BASE',      0.20, 2.0, 0.50, 5, C_LOG),
    ('ACELERADO', 0.25, 4.0, 0.90, 5, C_GOM),
    ('ÓPTIMO',    0.30, 6.0, 1.50, 5, C_EXP),
]

def phi_L(t, K, gamma, t0):
    return 1 + K / (1 + np.exp(-gamma * (t - t0)))

def V_model(t, C0, r, K, gamma, t0):
    phi0 = phi_L(0, K, gamma, t0)
    return C0 * (1 + r)**t * phi_L(t, K, gamma, t0) / phi0

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), facecolor=BG)
fig.suptitle('Modelo V(t) = Capital × (1+r)^t × Φ_L(t)/Φ_L(0)',
             color=WHITE, fontweight='bold', fontsize=12)

# Izquierda: Φ_L(t) por escenario
ax = axes[0]
ax.set_facecolor(BG)
ax.set_title('Φ_L(t) — curva de autorreplicación', color=WHITE, fontsize=10)
for name, r, K, g, t0, col in escenarios:
    phi0 = phi_L(0, K, g, t0)
    ax.plot(t, phi_L(t, K, g, t0) / phi0, color=col, lw=2, label=f'{name} K={K}')
ax.axvline(5, color='white', lw=0.8, ls='--', alpha=0.4, label='t₀=5 (inflexión)')
ax.legend(fontsize=8, facecolor=BG, labelcolor=WHITE)
ax.tick_params(colors=WHITE); [s.set_color(GRID) for s in ax.spines.values()]

# Derecha: V(t) por escenario
ax2 = axes[1]
ax2.set_facecolor(BG)
ax2.set_title('V(t) — proyección de capital (C₀=100.000€)', color=WHITE, fontsize=10)
for name, r, K, g, t0, col in escenarios:
    ax2.plot(t, V_model(t, 100_000, r, K, g, t0), color=col, lw=2, label=name)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'€{x/1e6:.1f}M' if x>=1e6 else f'€{x/1e3:.0f}K'))
ax2.legend(fontsize=8, facecolor=BG, labelcolor=WHITE)
ax2.tick_params(colors=WHITE); [s.set_color(GRID) for s in ax2.spines.values()]

plt.tight_layout()
plt.show()


## 3. El factor de autorreplicación Φ_L(t)

El modelo es:
$$V(t) = Capital \times (1+r)^t \times \frac{\Phi_L(t)}{\Phi_L(0)}$$

donde $(1+r)^t$ se calibra desde el SOX y $\Phi_L(t)$ es el **factor de autorreplicación logístico**:

$$\Phi_L(t) = 1 + \frac{K}{1 + e^{-\gamma(t - t_0)}}$$

| Parámetro | Descripción | Calibración |
|-----------|-------------|-------------|
| $K$ | Multiplicador máximo de autorreplicación | Escenario (2–6×) |
| $\gamma$ | Velocidad de adopción | `ln(2) / años_entre_releases` |
| $t_0$ | Año de inflexión (máxima aceleración) | Señal SOX↑/Cobre↓ |

La normalización por $\Phi_L(0)$ garantiza $V(0) = Capital$ para cualquier combinación de parámetros.


In [ ]:
# ── Definición del modelo logístico ────────────────────────────────────
def phi_L(t, K=4.0, gamma=0.9, t0=5.0):
    """Factor de autorreplicación logístico. K = techo, gamma = velocidad, t0 = inflexión."""
    return 1 + K / (1 + np.exp(-gamma * (t - t0)))

def V_model(t, C0, r, K, gamma, t0):
    """V(t) normalizado: V(0) = C0 para cualquier parámetro."""
    phi0 = phi_L(0, K, gamma, t0)
    return C0 * (1 + r)**t * phi_L(t, K, gamma, t0) / phi0

# Escenarios del paper (portfolio.yaml)
ESCENARIOS = [
    {'name': 'BASE',      'r': 0.20, 'K': 2.0, 'gamma': 0.50, 't0': 5, 'color': C_LOG},
    {'name': 'ACELERADO', 'r': 0.25, 'K': 4.0, 'gamma': 0.90, 't0': 5, 'color': C_GOM},
    {'name': 'ÓPTIMO',    'r': 0.30, 'K': 6.0, 'gamma': 1.50, 't0': 5, 'color': C_EXP},
]
print('Modelo logístico cargado ✓')


In [ ]:
# ── Visualización del factor Φ_L(t) por escenario ──────────────────────
t = np.linspace(0, 10, 300)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG)
fig.suptitle('Factor de autorreplicación Φ_L(t) — modelo Escasez y Resiliencia',
             color=WHITE, fontweight='bold', fontsize=12)

# Panel izquierdo: Φ_L(t)/Φ_L(0) normalizado
ax = axes[0]
ax.set_facecolor(PANEL)
ax.set_title('Φ_L(t)/Φ_L(0) por escenario', color=WHITE, fontsize=10)
for sc in ESCENARIOS:
    phi0 = phi_L(0, sc['K'], sc['gamma'], sc['t0'])
    ax.plot(t, phi_L(t, sc['K'], sc['gamma'], sc['t0']) / phi0,
            color=sc['color'], lw=2.5, label=f"{sc['name']}  K={sc['K']} γ={sc['gamma']}")
ax.axvline(5, color='white', lw=0.8, ls='--', alpha=0.4, label='t₀=5 (inflexión)')
ax.set_xlabel('Años', color=WHITE); ax.set_ylabel('Φ_L(t)/Φ_L(0)', color=WHITE)
ax.legend(fontsize=8, facecolor=BG, labelcolor=WHITE)
ax.tick_params(colors=WHITE); [s.set_color(GRID) for s in ax.spines.values()]

# Panel derecho: V(t) en € por escenario
ax2 = axes[1]
ax2.set_facecolor(PANEL)
ax2.set_title('V(t) — proyección de capital (C₀=100.000€)', color=WHITE, fontsize=10)
for sc in ESCENARIOS:
    ax2.plot(t, V_model(t, 100_000, sc['r'], sc['K'], sc['gamma'], sc['t0']),
             color=sc['color'], lw=2.5, label=sc['name'])
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'€{x/1e6:.1f}M' if x>=1e6 else f'€{x/1e3:.0f}K'))
ax2.set_xlabel('Años', color=WHITE)
ax2.legend(fontsize=9, facecolor=BG, labelcolor=WHITE)
ax2.tick_params(colors=WHITE); [s.set_color(GRID) for s in ax2.spines.values()]

plt.tight_layout(); plt.show()


## 4. Calibración de parámetros

### 4.1 Calibración de r desde el SOX

$$r_{IA} = \frac{\text{Rendimiento}_{SOX, 12M}}{100}$$

### 4.2 Calibración de λ (parámetro de autorreplicación)

Para el modelo logístico revisado, los parámetros a calibrar son:

- **K** = multiplicador de autorreplicación máximo. Interpretación: si K=4, la IA multiplica por 5 el potencial de crecimiento base en su punto de madurez.
- **γ** = velocidad de adopción (inversamente proporcional al tiempo de duplicación de la S-curve)
- **t₀** = año de inflexión (punto de máxima aceleración). Proxy: ¿cuándo se espera AGI o autofabricación robótica?


In [ ]:
# ── Calibración de γ desde la cadencia de releases LLM ──────────
# γ = ln(2) / (años que tarda la adopción en duplicar su tasa)

# Datos históricos aproximados de cadencia de releases LLM mayores:
llm_releases = {
    'GPT-3':   ('2020-06', 'OpenAI'),
    'GPT-3.5': ('2022-11', 'OpenAI'),
    'GPT-4':   ('2023-03', 'OpenAI'),
    'Claude 2':('2023-07', 'Anthropic'),
    'Llama 2': ('2023-07', 'Meta'),
    'GPT-4o':  ('2024-05', 'OpenAI'),
    'Claude 3':('2024-03', 'Anthropic'),
    'Llama 3': ('2024-04', 'Meta'),
    'o1':      ('2024-09', 'OpenAI'),
    'Claude 3.5':('2024-06','Anthropic'),
    'DeepSeek R1':('2025-01','DeepSeek'),
    'GPT-4.5': ('2025-02', 'OpenAI'),
    'Claude 3.7':('2025-02','Anthropic'),
    'Llama 4': ('2025-04', 'Meta'),
    'GPT-5':   ('2025-05', 'OpenAI'),
}

# Convertir a meses desde GPT-3
from datetime import datetime
def months_since(date_str, base='2020-06'):
    d = datetime.strptime(date_str, '%Y-%m')
    b = datetime.strptime(base, '%Y-%m')
    return (d.year - b.year)*12 + (d.month - b.month)

releases_months = sorted([months_since(v[0]) for v in llm_releases.values()])
print("Meses desde GPT-3 hasta cada release mayor:")
print(releases_months)

# Intervalos entre releases
intervals = np.diff(releases_months)
print(f"\nIntervalo medio: {np.mean(intervals):.1f} meses")
print(f"Intervalo mediana: {np.median(intervals):.1f} meses")
print(f"Intervalo reciente (últimos 5): {np.mean(intervals[-5:]):.1f} meses")

# Estimación de γ
# γ = velocidad de adopción ≈ 12 / intervalo_meses (para escala anual)
gamma_base = 12 / max(np.mean(intervals), 1)
gamma_acelerado = 12 / max(np.mean(intervals[-5:]), 1)
print(f"\nγ_base     (intervalo histórico): {gamma_base:.3f}")
print(f"γ_acelerado (intervalo reciente):  {gamma_acelerado:.3f}")


In [ ]:
# ── Visualización de la cadencia de releases ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), facecolor=BG)
fig.suptitle('Calibración de γ — cadencia de releases LLM',
             color=WHITE, fontweight='bold')

ax = axes[0]
ax.set_facecolor(PANEL)
ax.bar(range(len(releases_months)), releases_months,
       color=[C_LOG if i < len(releases_months)-5 else C_OPT
              for i in range(len(releases_months))], alpha=0.8)
ax.set_xticks(range(len(releases_months)))
ax.set_xticklabels([k[:8] for k in llm_releases.keys()], rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Meses desde GPT-3')
ax.set_title('Releases LLM acumulados desde 2020', color=WHITE)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.set_facecolor(PANEL)
ax.bar(range(len(intervals)), intervals,
       color=[C_LOG if i < len(intervals)-5 else C_OPT for i in range(len(intervals))],
       alpha=0.8)
ax.axhline(np.mean(intervals), color=C_BASE, lw=1.5, ls='--',
           label=f'Media: {np.mean(intervals):.1f}m')
ax.axhline(np.mean(intervals[-5:]), color=C_OPT, lw=1.5, ls='--',
           label=f'Media reciente: {np.mean(intervals[-5:]):.1f}m')
ax.set_ylabel('Meses entre releases')
ax.set_title('Intervalo entre releases consecutivos', color=WHITE)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Calibración recomendada:")
print(f"   γ_base     = {gamma_base:.2f}  (escenario base)")
print(f"   γ_acelerado = {gamma_acelerado:.2f}  (escenario acelerado)")


## 5. Propiedades del modelo logístico


In [ ]:
# ── Tabla de propiedades del modelo logístico ──────────────────────────
import pandas as pd

propiedades = {
    'Propiedad': [
        'Fórmula',
        'Saturación',
        'Valor en t=0',
        'Falsificable',
        'Punto de inflexión',
        'Conexión SOX/Cobre',
        'Parámetros'
    ],
    'Modelo logístico V(t) = Capital × (1+r)^t × Φ_L(t)/Φ_L(0)': [
        'Φ_L(t) = 1 + K / (1 + e^(−γ(t−t₀)))',
        '✓  K+1  (acotado)',
        '✓  V(0) = Capital  (normalizado)',
        '✓  Si IA se detiene → Φ_L→1 → V=(1+r)^t',
        '✓  t₀  (calibrable desde SOX/Cobre)',
        '✓  t₀ = año de Divergencia SOX↑/Cobre↓',
        'r, K, γ, t₀  (independientes)'
    ]
}

df = pd.DataFrame(propiedades)
print(df.to_string(index=False))


## 6. Backtesting con datos históricos del portfolio

Usamos los datos reales disponibles para validar qué modelo se ajusta mejor al comportamiento histórico de los activos del portfolio.


In [ ]:
import os, csv
from datetime import datetime

HIST_DIR = os.path.join(os.path.dirname(os.getcwd()),
           'mnt', 'Cartera-Inversion', 'historico')

def load_csv(isin, hist_dir=HIST_DIR):
    """Carga CSV histórico de un activo."""
    path = os.path.join(hist_dir, f'{isin}.csv')
    if not os.path.exists(path):
        return None, None
    dates, prices = [], []
    with open(path) as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                dates.append(datetime.strptime(row['fecha'], '%Y-%m-%d'))
                prices.append(float(row['precio_cierre']))
            except:
                pass
    return dates, prices

# Cargar activos representativos de cada pilar
activos = {
    'SOX (proxy IA)':       'IE00BGV5VN51',   # AI & Big Data
    'Robótica':             'IE00BYZK4552',   # Automatización
    'Uranio/Nuclear':       'IE000M7V94E1',   # Energía
    'Cobre':                'GB00B15KXQ89',   # Metal
    'Bitcoin':              'BTC-EUR',        # Escasez digital
    'MSCI World':           'IE00B03HCZ61',   # Resiliencia
}

data = {}
for nombre, isin in activos.items():
    dates, prices = load_csv(isin)
    if dates and len(dates) > 50:
        data[nombre] = (dates, prices)
        print(f"✅ {nombre:25s} | {isin} | {len(dates)} registros | "
              f"{dates[0].strftime('%Y-%m-%d')} → {dates[-1].strftime('%Y-%m-%d')}")
    else:
        print(f"⚠️  {nombre:25s} | Sin datos suficientes")


In [ ]:
# ── Calcular retornos acumulados normalizados ────────────────────
if data:
    fig, ax = plt.subplots(figsize=(14, 6), facecolor=BG)
    ax.set_facecolor(PANEL)

    colors_map = [C_LOG, C_GOM, C_OPT, C_EXP, C_BASE, C_DUAL]
    for (nombre, (dates, prices)), color in zip(data.items(), colors_map):
        # Normalizar desde el primer precio disponible
        p_arr = np.array(prices)
        d_arr = np.array([(d - dates[0]).days / 365.25 for d in dates])
        p_norm = p_arr / p_arr[0]

        ax.plot(d_arr, p_norm, lw=1.8, color=color, label=nombre, alpha=0.9)
        # Anotar valor final
        ax.annotate(f'{p_norm[-1]:.1f}×', xy=(d_arr[-1], p_norm[-1]),
                    fontsize=7.5, color=color, xytext=(d_arr[-1]+0.05, p_norm[-1]),
                    va='center')

    ax.axhline(1, color=MUTED, lw=0.8, ls=':', alpha=0.5)
    ax.set_xlabel('Años desde inicio del histórico')
    ax.set_ylabel('Precio normalizado (1 = precio inicial)')
    ax.set_title('Retornos acumulados reales — activos del portfolio',
                 color=WHITE, fontweight='bold')
    ax.legend(fontsize=8.5, loc='upper left')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  No se encontraron datos históricos. Asegúrate de ejecutar desde el directorio del proyecto.")


In [ ]:
# ── Fit del modelo logístico a los datos reales ──────────────────
# Fit manual por mínimos cuadrados para un activo

def logistic_model(t, K, gamma, t0, r=0.20, C0=1.0):
    """V(t) = C0 × (1+r)^t × (1 + K/(1+e^(-γ(t-t0))))"""
    return C0 * (1+r)**t * (1 + K / (1 + np.exp(-gamma * (t - t0))))

def mse(params, t_data, y_data):
    K, gamma, t0 = params
    if K < 0 or gamma < 0 or t0 < 0: return 1e10
    y_pred = logistic_model(t_data, K, gamma, t0)
    return np.mean((y_data - y_pred)**2)

def gradient_descent_fit(t_data, y_data, n_iter=5000, lr=0.001):
    """Fit por descenso de gradiente (sin scipy)."""
    K, gamma, t0 = 2.0, 0.5, np.mean(t_data)
    best_params = (K, gamma, t0)
    best_loss = mse((K, gamma, t0), t_data, y_data)

    for _ in range(n_iter):
        eps = 1e-5
        dK  = (mse((K+eps, gamma, t0), t_data, y_data) - best_loss) / eps
        dg  = (mse((K, gamma+eps, t0), t_data, y_data) - best_loss) / eps
        dt0 = (mse((K, gamma, t0+eps), t_data, y_data) - best_loss) / eps
        K     = max(0.01, K  - lr * dK)
        gamma = max(0.01, gamma - lr * dg)
        t0    = max(0.1,  t0  - lr * dt0)
        loss  = mse((K, gamma, t0), t_data, y_data)
        if loss < best_loss:
            best_loss = loss
            best_params = (K, gamma, t0)
    return best_params

if data:
    # Usar el activo con más histórico para el fit
    nombre_ref = max(data.keys(), key=lambda k: len(data[k][0]))
    dates_ref, prices_ref = data[nombre_ref]
    t_ref = np.array([(d - dates_ref[0]).days / 365.25 for d in dates_ref])
    y_ref = np.array(prices_ref) / prices_ref[0]  # normalizado

    print(f"Ajustando modelo logístico a: {nombre_ref}")
    print(f"Registros: {len(t_ref)}, rango: {t_ref[0]:.2f}–{t_ref[-1]:.2f} años")

    K_fit, g_fit, t0_fit = gradient_descent_fit(t_ref, y_ref)

    t_smooth = np.linspace(0, max(t_ref)*1.3, 300)
    y_pred = logistic_model(t_smooth, K_fit, g_fit, t0_fit)

    fig, ax = plt.subplots(figsize=(12, 5), facecolor=BG)
    ax.set_facecolor(PANEL)
    ax.scatter(t_ref[::5], y_ref[::5], color=C_BASE, s=10, alpha=0.4, label='Datos reales (1/5)')
    ax.plot(t_smooth, y_pred, color=C_LOG, lw=2.5,
            label=f'Modelo logístico fit: K={K_fit:.2f}, γ={g_fit:.2f}, t₀={t0_fit:.1f}')
    ax.axvline(t0_fit, color=C_OPT, lw=1, ls=':', alpha=0.7)
    ax.text(t0_fit+0.05, 0.3, f't₀={t0_fit:.1f}a', fontsize=8, color=C_OPT)
    ax.set_xlabel('Años desde inicio'); ax.set_ylabel('Precio normalizado')
    ax.set_title(f'Fit logístico — {nombre_ref}', color=WHITE, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    print(f"\nParámetros ajustados:")
    print(f"  K  = {K_fit:.3f}  (multiplicador de autorreplicación)")
    print(f"  γ  = {g_fit:.3f}  (velocidad de adopción)")
    print(f"  t₀ = {t0_fit:.2f} años desde inicio del histórico")
    print(f"  → Correlación con calibración teórica: γ_base={gamma_base:.2f}, γ_acel={gamma_acelerado:.2f}")
else:
    print("Sin datos. Ejecuta con acceso a /historico/")


## 7. Análisis de sensibilidad — ¿qué parámetro importa más?

In [ ]:
# Análisis de sensibilidad del modelo logístico
C0, t_horizon = 100_000, 10
t = np.linspace(0, t_horizon, 200)

fig, axes = plt.subplots(2, 3, figsize=(16, 9), facecolor=BG)
fig.suptitle('Análisis de sensibilidad — Modelo Logístico V(t) = C₀×(1+r)^t×Φ_L(t)',
             color=WHITE, fontweight='bold', fontsize=12)

params_defaults = dict(r=0.20, K=3.0, gamma=0.9, t0=5.0)

sweep = {
    'r (crecimiento IA)':     ('r',     [0.08, 0.12, 0.15, 0.20, 0.25, 0.30]),
    'K (multiplicador max)':  ('K',     [1.0, 2.0, 3.0, 4.0, 6.0, 9.0]),
    'γ (velocidad adopción)': ('gamma', [0.3, 0.5, 0.7, 0.9, 1.2, 1.8]),
    't₀ (año inflexión)':     ('t0',    [2.0, 3.5, 5.0, 6.5, 8.0, 9.5]),
    'r + K combinados':       None,
    'Resumen: V a 10 años':   None,
}

colors_sweep = ['#1f6feb','#388bfd','#58a6ff','#79c0ff','#a5d6ff','#cae8ff']

for ax, (label, spec) in zip(axes.flat, sweep.items()):
    ax.set_facecolor(PANEL)

    if spec is not None:
        param_name, values = spec
        for val, col in zip(values, colors_sweep):
            p = dict(params_defaults)
            p[param_name] = val
            V = C0 * (1+p['r'])**t * phi_logistic(t, p['K'], p['gamma'], p['t0'])
            ax.plot(t, V/1e6, color=col, lw=1.8, label=f'{param_name}={val}')
        ax.set_xlabel('Años'); ax.set_ylabel('M€')
        ax.set_title(label, color=WHITE, fontsize=9)
        ax.legend(fontsize=7, loc='upper left')
        ax.set_xticks(range(0, 11, 2))
        ax.set_xticklabels([str(2026+v) for v in range(0, 11, 2)], fontsize=7)
        ax.grid(True, alpha=0.3)

    elif 'combinados' in label:
        for r_v, col, ls in zip([0.15, 0.20, 0.25], [C_GOM, C_LOG, C_OPT], ['-','--',':']):
            for K_v, alpha in zip([2.0, 4.0], [0.6, 1.0]):
                V = C0 * (1+r_v)**t * phi_logistic(t, K_v, 0.9, 5.0)
                ax.plot(t, V/1e6, color=col, lw=1.8, ls=ls, alpha=alpha,
                        label=f'r={r_v}, K={K_v}')
        ax.set_xlabel('Años'); ax.set_ylabel('M€')
        ax.set_title('r × K combinados', color=WHITE, fontsize=9)
        ax.legend(fontsize=6.5, loc='upper left')
        ax.set_xticks(range(0, 11, 2))
        ax.set_xticklabels([str(2026+v) for v in range(0, 11, 2)], fontsize=7)
        ax.grid(True, alpha=0.3)

    else:
        # Resumen: V a 10 años para cada parámetro
        param_labels, impacts = [], []
        for pname, pvals in [('r', [0.10, 0.20, 0.30]),
                              ('K', [1.0, 3.0, 6.0]),
                              ('γ', [0.3, 0.9, 1.8]),
                              ('t₀', [2.0, 5.0, 9.0])]:
            lo, mid, hi = pvals
            def V10(pn, pv):
                p = dict(params_defaults)
                pmap = {'r':'r','K':'K','γ':'gamma','t₀':'t0'}
                p[pmap[pn]] = pv
                return C0 * (1+p['r'])**10 * phi_logistic(np.array([10.0]), p['K'], p['gamma'], p['t0'])[0]
            rng = (V10(pname, hi) - V10(pname, lo)) / 1e6
            impacts.append(rng)
            param_labels.append(pname)

        bars = ax.barh(param_labels, impacts, color=[C_LOG, C_OPT, C_GOM, C_BASE])
        ax.set_xlabel('Rango V(10) en M€')
        ax.set_title('Impacto de cada parámetro (hi−lo)', color=WHITE, fontsize=9)
        ax.grid(True, alpha=0.3)
        for bar, impact in zip(bars, impacts):
            ax.text(impact+0.05, bar.get_y()+bar.get_height()/2,
                    f'{impact:.1f}M€', va='center', fontsize=8, color=WHITE)

plt.tight_layout()
plt.show()

print("\nConclusion del análisis de sensibilidad:")
print(f"  El parámetro con mayor impacto es K (multiplicador de autorreplicación),")
print(f"  seguido de r. El parámetro t₀ tiene impacto medio — pero es el más")
print(f"  importante para la TIMING de la rotación táctica (detector SOX/Cobre).")


## 8. Simulación del detector SOX/Cobre

El modelo de Régimen Dual vincula matemáticamente la señal de mercado (Divergencia SOX/Cobre) con el cambio de parámetros del modelo. Esta sección simula diferentes escenarios de cuándo ocurre la Divergencia.


In [ ]:
# Simulación: ¿qué pasa si la Divergencia ocurre en distintos años?
C0, r = 100_000, 0.20
t = np.linspace(0, 10, 300)

t0_divergencias = [3, 5, 7, 9, None]  # None = nunca ocurre
labels_div = [f'Divergencia t={t0}' for t0 in t0_divergencias[:-1]] + ['Sin divergencia']
lam1, lam2 = 0.25, 0.02  # λ alto → λ bajo

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), facecolor=BG)
fig.suptitle('Detector SOX/Cobre — impacto del timing de la Divergencia',
             color=WHITE, fontweight='bold')

# Panel izquierdo: V(t) según timing de divergencia
ax = axes[0]
ax.set_facecolor(PANEL)
colors_div = ['#ff7b72','#ffa657','#e3b341','#56d364','#79c0ff']
for t0, label, col in zip(t0_divergencias, labels_div, colors_div):
    if t0 is None:
        V = C0 * (1+r)**t * phi_exp(t, lam=lam1)
        ax.plot(t, V/1e6, color=col, lw=2.5, ls='--', label=label)
    else:
        V = C0 * (1+r)**t * phi_dual(t, lam1=lam1, lam2=lam2, t0=float(t0))
        ax.plot(t, V/1e6, color=col, lw=1.8, label=label)
        ax.axvline(t0, color=col, lw=0.8, ls=':', alpha=0.4)

ax.set_xlabel('Años'); ax.set_ylabel('Valor cartera (M€)')
ax.set_title('V(t) según año de la señal de Divergencia', color=WHITE)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax.set_xticks(range(0, 11, 2))
ax.set_xticklabels([str(2026+v) for v in range(0, 11, 2)], fontsize=8)

# Panel derecho: valor final a 10 años según t0
ax = axes[1]
ax.set_facecolor(PANEL)
t0_range = np.linspace(1, 9.9, 50)
V_finals = []
for t0 in t0_range:
    V = C0 * (1+r)**10 * phi_dual(np.array([10.0]), lam1=lam1, lam2=lam2, t0=t0)[0]
    V_finals.append(V/1e6)

ax.plot(t0_range, V_finals, color=C_LOG, lw=2.5)
ax.fill_between(t0_range, V_finals,
                [C0*(1+r)**10/1e6]*len(t0_range),
                alpha=0.15, color=C_LOG)
ax.axhline(C0*(1+r)**10/1e6, color=C_BASE, lw=1, ls=':',
           label=f'Sin autorrepl.: €{C0*(1+r)**10/1e6:.1f}M')

# Marcar el punto óptimo
idx_max = np.argmax(V_finals)
ax.scatter([t0_range[idx_max]], [V_finals[idx_max]], color=C_OPT, s=80, zorder=5)
ax.annotate(f'Máximo
€{V_finals[idx_max]:.1f}M
t₀={t0_range[idx_max]:.1f}',
            xy=(t0_range[idx_max], V_finals[idx_max]),
            xytext=(t0_range[idx_max]+0.5, V_finals[idx_max]*0.85),
            color=C_OPT, fontsize=8,
            arrowprops=dict(arrowstyle='->', color=C_OPT))

ax.set_xlabel('Año de la Divergencia SOX/Cobre')
ax.set_ylabel('Valor a 10 años (M€)')
ax.set_title('Valor final vs timing de la Divergencia', color=WHITE)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax.set_xticks(range(1, 11, 1))
ax.set_xticklabels([str(2026+v) for v in range(1, 11)], fontsize=7.5, rotation=30)

plt.tight_layout()
plt.show()

print("\n💡 Insight: Cuanto más tarde ocurre la Divergencia, mayor el valor acumulado.")
print("   La señal SOX/Cobre es un indicador TÁCTICO de rotación, no de salida total.")


## 9. Proyecciones finales — 3 escenarios


In [ ]:
from datetime import date

today_yr = date.today().year

# Cargar capital actual del portfolio si está disponible
try:
    import yaml, os
    YAML_PATH = os.path.join(os.path.dirname(os.getcwd()),
                'mnt', 'Cartera-Inversion', 'portfolio.yaml')
    with open(YAML_PATH) as f:
        pf = yaml.safe_load(f)
    # Calcular valor estimado desde precios_actuales
    activos_yaml = pf.get('activos', [])
    C0_real = sum(
        float(a.get('precios_actuales', {}).get('precio', 0) or 0) *
        sum(float(tx.get('participaciones', 0) or 0)
            for tx in a.get('transacciones', [])
            if tx.get('tipo') == 'compra')
        for a in activos_yaml
    )
    if C0_real > 1000:
        print(f"✅ Capital estimado desde portfolio.yaml: €{C0_real:,.0f}")
    else:
        C0_real = 100_000
        print(f"⚠️  No se pudo calcular capital exacto — usando €100.000 de referencia")
except Exception as e:
    C0_real = 100_000
    print(f"⚠️  Usando C0=€100.000 de referencia ({e})")

C0 = C0_real
t  = np.linspace(0, 10, 300)
horizons = [2, 4, 6, 8, 10]

# Proyección por escenario
escenarios = [
    ('Base',      B_r, B_lam, C_BASE,   0.05, 3.0, 0.5, 5.0),
    ('Acelerado', A_r, A_lam, C_OPT,    0.15, 5.0, 0.9, 4.0),
    ('Óptimo',    O_r, O_lam, C_LOG,    0.30, 8.0, 1.4, 3.0),
]
# (nombre, r, lam_orig, color, lam_revisado, K, gamma, t0)

fig, axes = plt.subplots(1, 3, figsize=(16, 6), facecolor=BG)
fig.suptitle(f'Proyecciones {today_yr}→{today_yr+10}  —  3 Escenarios  |  Capital: €{C0:,.0f}',
             color=WHITE, fontweight='bold', fontsize=12)

print(f"\n{'Escenario':<12} | {'Modelo':<12} | " +
      " | ".join(f"+{h}a" for h in horizons))
print("─" * 80)

for ax, (nombre, r_s, lam_orig, col, lam_rev, K, gamma, t0) in zip(axes, escenarios):
    ax.set_facecolor(PANEL)

    V_orig = C0 * (1+r_s)**t * phi_exp(t, lam=lam_orig)
    V_log  = C0 * (1+r_s)**t * phi_logistic(t, K=K, gamma=gamma, t0=t0)
    V_base_only = C0 * (1+r_s)**t

    ax.fill_between(t, V_log/1e6, V_base_only/1e6, alpha=0.12, color=col)
    ax.plot(t, V_base_only/1e6, color=MUTED, lw=1.2, ls=':', label='Solo (1+r)^t')
    ax.plot(t, V_log/1e6,  color=col,     lw=2.5, label=f'Logístico (K={K}, t₀={t0})')
    ax.plot(t, V_orig/1e6, color=C_EXP,   lw=1.8, ls='--', label=f'Exponencial (λ={lam_orig})')

    # Marcar horizontes
    for h in horizons:
        v_l = C0 * (1+r_s)**h * phi_logistic(np.array([float(h)]), K=K, gamma=gamma, t0=t0)[0]
        v_e = C0 * (1+r_s)**h * phi_exp(np.array([float(h)]), lam=lam_orig)[0]
        ax.scatter([h], [v_l/1e6], color=col, s=35, zorder=5)

    ax.set_title(f'Escenario {nombre}
r={r_s:.0%}, K={K}, t₀={t0}',
                 color=WHITE, fontsize=9)
    ax.set_xlabel('Años'); ax.set_ylabel('M€')
    ax.legend(fontsize=7.5); ax.grid(True, alpha=0.3)
    ax.set_xticks(range(0, 11, 2))
    ax.set_xticklabels([str(today_yr+v) for v in range(0, 11, 2)], fontsize=7.5)

    # Tabla resumen
    for modelo, phi_fn, args in [('Original ', phi_exp,      {'lam': lam_orig}),
                                   ('Logístico', phi_logistic, {'K': K, 'gamma': gamma, 't0': t0})]:
        vals = [C0 * (1+r_s)**h * phi_fn(np.array([float(h)]), **args)[0] for h in horizons]
        row  = " | ".join(f"€{v/1e6:.1f}M" for v in vals)
        print(f"{nombre:<12} | {modelo:<12} | {row}")

print("─" * 80)
plt.tight_layout()
plt.show()


## 10. Conclusiones del laboratorio

### Modelo del paper (Vilar, 2026):

$$V(t) = Capital \times (1+r_{IA})^t \times \frac{\Phi_L(t)}{\Phi_L(0)}$$

$$\Phi_L(t) = 1 + \frac{K}{1 + e^{-\gamma(t - t_0)}}$$

**Parámetros y su calibración:**
| Parámetro | Descripción | Fuente de calibración |
|-----------|-------------|----------------------|
| $r_{IA}$ | Crecimiento del ecosistema IA | SOX trailing 12M / 100 |
| $K$ | Multiplicador máximo de autorreplicación | Escenario (2–6×) |
| $\gamma$ | Velocidad de adopción | `ln(2) / (meses_entre_releases / 12)` |
| $t_0$ | Año de inflexión (máxima aceleración) | Señal Divergencia SOX↑/Cobre↓ |

**Referencia:** Verhulst, P.-F. (1838). Notice sur la loi que la population suit dans son accroissement. *Correspondance Mathématique et Physique*, 10, 113–121.

**Paper completo:** Vilar, J. A. (2026). *Scarcity and Resilience: A Mathematical Framework for Investing in the Age of Artificial Self-Replication*. SSRN Working Paper.
